<a href="https://colab.research.google.com/github/nileve-teo/Metodos-numericos/blob/main/Integraci%C3%B3n_num%C3%A9rica2_9_de_Junio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="
width:23cm;
max-width:23cm;
box-sizing:border-box;
margin:auto;
background-color:#E3D5B4;
border-top:2px solid #444;
padding-top:18px;
padding-bottom:18px;
padding-left:20px;
padding-right:20px;
font-family:Georgia;
overflow:hidden;
">

<div style="
text-align:left;
font-size:42px;
color:#1f1f1f;
margin-bottom:12px;
">
INTEGRACIÓN NUMÉRICA
</div>

<div style="
width:100%;
box-sizing:border-box;
background-color:#2E6599;
color:white;
font-size:38px;
padding:20px 30px;
text-align:right;
">
Gauss-Legendre: Nodos Óptimos
</div>

</div>

<div style="
width:23cm;
max-width:23cm;
box-sizing:border-box;
margin:auto;
background-color:#F2F0E6;
border-top:2px solid #444;
padding:20px;
font-family:Georgia;
overflow:hidden;
line-height:1.7;
">

<h2 style="
color:#2E6599;
font-family:'Avenir Arabic Roman','Avenir','Helvetica Neue',Arial,sans-serif;
font-weight:700;
">
Ejercicio 1 – Clase de Laboratorio
</h2>

Desarrolle un **script en Python** para resolver las siguientes integrales.

<div style="
background-color:#ECE9DC;
padding:15px;
border-radius:8px;
text-align:center;
margin-top:15px;
margin-bottom:15px;
">

$$
f(x)=\frac{1}{\sqrt{2\pi}}e^{-x^2/2}
$$

$$
f(x)=\cos(x^2)
$$

<br>

$$
f(x)=x^5-2x^3+4
$$

</div>

Optimice su código para que la función pueda ingresarse desde pantalla y no sea necesario modificar el programa constantemente.

Además, el código debe:

- Identificar automáticamente si la función es **polinómica**.
- Determinar automáticamente el número óptimo de nodos para **Gauss-Legendre**.
- Obtener la solución exacta cuando sea posible.

**Nota:** Al utilizar Cuadratura de Gauss-Legendre de \(n\) puntos, las raíces y los pesos deben calcularse de forma **algorítmica**.

<h2 style="
color:#2E6599;
font-family:'Avenir Arabic Roman','Avenir','Helvetica Neue',Arial,sans-serif;
font-weight:bold;
">
Métodos Numéricos Requeridos
</h2>

- **Regla del Trapecio Compuesta** (N = 12 subintervalos)
- **Regla de Simpson 1/3 Compuesta** (N = 12 subintervalos)
- **Regla de Simpson 3/8 Compuesta** (N = 12 subintervalos)
- **Cuadratura de Gauss-Legendre**
  - Utilizar únicamente **n = 5** para las dos primeras integrales.
  - Escoger automáticamente el valor óptimo de **n** para la tercera integral.

<h2 style="
color:#2E6599;
font-family:'Avenir Arabic Roman','Avenir','Helvetica Neue',Arial,sans-serif;
font-weight:700;
">
Presentación de Resultados
</h2>

| Método | Puntos F(x) | Aproximación | Error Absoluto |
|----------|----------|----------|----------|
| Trapecio Compuesto | | | |
| Simpson 1/3 Compuesto | | | |
| Simpson 3/8 Compuesto | | | |
| Gauss-Legendre | | | |


</div>

In [24]:
# ============================================================
# LABORATORIO 1 - INTEGRACIÓN NUMÉRICA
# ============================================================

import numpy as np
import pandas as pd
import sympy as sp
from IPython.display import display, Math

x = sp.Symbol('x')

# ============================================================
# TRAPECIO COMPUESTO
# ============================================================

def trapecio_compuesto(f, a, b, N=12):

    X = np.linspace(a, b, N + 1)
    Y = f(X)

    h = (b - a) / N

    I = h * (
        0.5 * Y[0]
        + np.sum(Y[1:-1])
        + 0.5 * Y[-1]
    )

    return float(I), len(X)

# ============================================================
# SIMPSON 1/3 COMPUESTO
# ============================================================

def simpson13_compuesto(f, a, b, N=12):

    X = np.linspace(a, b, N + 1)
    Y = f(X)

    h = (b - a) / N

    I = (h/3) * (
        Y[0]
        + Y[-1]
        + 4*np.sum(Y[1:-1:2])
        + 2*np.sum(Y[2:-2:2])
    )

    return float(I), len(X)

# ============================================================
# SIMPSON 3/8 COMPUESTO
# ============================================================

def simpson38_compuesto(f, a, b, N=12):

    X = np.linspace(a, b, N + 1)
    Y = f(X)

    h = (b - a) / N

    suma = 0

    for i in range(1, N):

        if i % 3 == 0:
            suma += 2 * Y[i]
        else:
            suma += 3 * Y[i]

    I = (3*h/8) * (Y[0] + Y[-1] + suma)

    return float(I), len(X)

# ============================================================
# GAUSS LEGENDRE ALGORÍTMICO
# ============================================================

def gauss_legendre_algoritmico(expr, a, b, n):

    t = sp.Symbol('t')

    Pn = sp.legendre(n, t)

    raices = [float(r) for r in sp.nroots(Pn)]

    dPn = sp.diff(Pn, t)

    pesos = []

    for r in raices:

        dp = float(dPn.subs(t, r))

        w = 2 / ((1-r**2)*(dp**2))

        pesos.append(w)

    f = sp.lambdify(x, expr, "numpy")

    suma = 0

    for xi, wi in zip(raices, pesos):

        xp = ((b-a)/2)*xi + (a+b)/2

        suma += wi * f(xp)

    I = ((b-a)/2)*suma

    return float(I), n

# ============================================================
# DETECTAR POLINOMIO
# ============================================================

def detectar_grado(expr):

    try:

        grado = sp.Poly(expr, x).degree()

        return True, grado

    except:

        return False, None

# ============================================================
# INTEGRAL EXACTA
# ============================================================

def integral_exacta(expr, a, b):

    return sp.integrate(expr, (x, a, b))

# ============================================================
# CASOS DEL LABORATORIO
# ============================================================

casos = [

    {
        "nombre":"Integral 1",
        "expr":1/sp.sqrt(2*sp.pi)*sp.exp(-x**2/2),
        "a":0,
        "b":3,
        "n_gauss":5
    },

    {
        "nombre":"Integral 2",
        "expr":sp.cos(x**2),
        "a":0,
        "b":2,
        "n_gauss":5
    },

    {
        "nombre":"Integral 3",
        "expr":x**5 - 2*x**3 + 4,
        "a":-2,
        "b":3,
        "n_gauss":"auto"
    }

]

# ============================================================
# RESOLUCIÓN
# ============================================================

for caso in casos:

    expr = caso["expr"]
    a = caso["a"]
    b = caso["b"]

    exacta_simbolica = integral_exacta(expr, a, b)
    exacta = float(exacta_simbolica.evalf())

    f = sp.lambdify(x, expr, "numpy")

    es_pol, grado = detectar_grado(expr)

    if caso["n_gauss"] == "auto":

        n = int(np.ceil((grado + 1)/2))

    else:

        n = caso["n_gauss"]

    It, p1 = trapecio_compuesto(f, a, b)

    Is13, p2 = simpson13_compuesto(f, a, b)

    Is38, p3 = simpson38_compuesto(f, a, b)

    Ig, p4 = gauss_legendre_algoritmico(
        expr, a, b, n
    )

    # ========================================================
    # DESARROLLO ANALÍTICO
    # ========================================================

    print("\n")
    print("="*80)
    print(caso["nombre"])
    print("="*80)

    F = sp.integrate(expr, x)

    print("\nFUNCIÓN:")

    display(
        Math(
            r"f(x)=" + sp.latex(expr)
        )
    )

    print("\nINTEGRAL DEFINIDA:")

    display(
        Math(
            r"\int_{%s}^{%s}%s\,dx"
            %
            (
                sp.latex(a),
                sp.latex(b),
                sp.latex(expr)
            )
        )
    )

    # ========================================================
    # POLINOMIOS
    # ========================================================

    if es_pol:

        print("\nINTEGRACIÓN:")

        display(
            Math(
                r"\int %s\,dx=%s"
                %
                (
                    sp.latex(expr),
                    sp.latex(F)
                )
            )
        )

        print("\nEVALUANDO LOS LÍMITES:")

        display(
            Math(
                r"\left[%s\right]_{%s}^{%s}"
                %
                (
                    sp.latex(F),
                    sp.latex(a),
                    sp.latex(b)
                )
            )
        )

        Fa = sp.simplify(F.subs(x, a))
        Fb = sp.simplify(F.subs(x, b))

        display(
            Math(
                r"=\left(%s\right)-\left(%s\right)"
                %
                (
                    sp.latex(Fb),
                    sp.latex(Fa)
                )
            )
        )

        display(
            Math(
                r"=%s"
                %
                sp.latex(exacta_simbolica)
            )
        )

    # ========================================================
    # DISTRIBUCIÓN NORMAL
    # ========================================================

    elif expr.has(sp.exp):

        print("\nSOLUCIÓN ANALÍTICA:")

        print(
            "La integral se expresa mediante "
            "la función error erf(x)."
        )

        display(
            Math(
                r"F(x)=\frac12"
                r"\operatorname{erf}"
                r"\left(\frac{x}{\sqrt2}\right)"
            )
        )

        display(
            Math(
                r"\left[\frac12"
                r"\operatorname{erf}"
                r"\left(\frac{x}{\sqrt2}\right)"
                r"\right]_{0}^{3}"
            )
        )

        display(
            Math(
                r"=\frac12"
                r"\operatorname{erf}"
                r"\left(\frac3{\sqrt2}\right)"
            )
        )

    # ========================================================
    # FRESNEL
    # ========================================================

    else:

        print("\nSOLUCIÓN ANALÍTICA:")

        print(
            "La integral no posee una "
            "antiderivada elemental."
        )

        print(
            "Se expresa mediante funciones "
            "especiales de Fresnel."
        )

        display(
            Math(
                r"\int_0^2 \cos(x^2)\,dx"
            )
        )

    print(f"\nVALOR EXACTO = {exacta:.12f}")

    if es_pol:

        print(f"\nPolinomio de grado = {grado}")
        print(f"n óptimo Gauss-Legendre = {n}")

    # ========================================================
    # TABLA FINAL
    # ========================================================

    tabla = pd.DataFrame({

        "Método":[
            "Valor Exacto",
            "Trapecio Compuesto",
            "Simpson 1/3 Compuesto",
            "Simpson 3/8 Compuesto",
            "Gauss-Legendre"
        ],

        "Puntos F(x)":[
            "-",
            p1,
            p2,
            p3,
            p4
        ],

        "Aproximación":[
            exacta,
            It,
            Is13,
            Is38,
            Ig
        ],

        "Error Absoluto":[
            0,
            abs(exacta-It),
            abs(exacta-Is13),
            abs(exacta-Is38),
            abs(exacta-Ig)
        ]

    })

    display(
        tabla.style
        .hide(axis="index")
        .format({
            "Aproximación":"{:.10f}",
            "Error Absoluto":"{:.3e}"
        })
    )



Integral 1

FUNCIÓN:


<IPython.core.display.Math object>


INTEGRAL DEFINIDA:


<IPython.core.display.Math object>


SOLUCIÓN ANALÍTICA:
La integral se expresa mediante la función error erf(x).


...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


VALOR EXACTO = 98.333333333333

Polinomio de grado = 5
n óptimo Gauss-Legendre = 3


Método,Puntos F(x),Aproximación,Error Absoluto
Valor Exacto,-,98.3333333333,0.000e+00
Trapecio Compuesto,13,102.5887144740,4.255e+00
Simpson 1/3 Compuesto,13,98.3835680298,5.023e-02
Simpson 3/8 Compuesto,13,98.4463614005,1.130e-01
Gauss-Legendre,3,98.3333333333,1.421e-14


<div style="
width:23cm;
max-width:23cm;
box-sizing:border-box;
margin:auto;
background-color:#F2F0E6;
border-top:2px solid #444;
padding:20px;
font-family:Georgia;
overflow:hidden;
line-height:1.7;
">

<h2 style="
color:#2E6599;
font-family:'Avenir Arabic Roman','Avenir','Helvetica Neue',Arial,sans-serif;
font-weight:700;
">
Análisis de Resultados </h2>

### **1. La Paradoja de los Puntos**

Los métodos de **Trapecio Compuesto** y **Simpson Compuesto** utilizaron un total de **13 evaluaciones de la función**, debido a que se emplearon **12 subintervalos**, generando 13 nodos de cálculo.

Por otro lado, la **Cuadratura de Gauss-Legendre** requirió únicamente **5 evaluaciones de la función**, correspondientes a los 5 nodos óptimos seleccionados por el método.

A pesar de utilizar menos puntos de evaluación, Gauss-Legendre alcanzó una precisión significativamente superior a los métodos de Newton-Cotes.

---

### **2. Análisis del Error**

Los métodos de Newton-Cotes emplean puntos equiespaciados en todo el intervalo de integración.

En cambio, Gauss-Legendre selecciona nodos ubicados estratégicamente, correspondientes a las raíces de los polinomios de Legendre.

Esta distribución permite maximizar el grado de exactitud de la cuadratura, reduciendo considerablemente el error numérico.

Por esta razón, aun utilizando menos evaluaciones de la función, Gauss-Legendre puede obtener errores varios órdenes de magnitud menores que los obtenidos mediante Trapecio o Simpson.

---

### **3. Comparación de Eficiencia**

La eficiencia de un método numérico puede evaluarse considerando simultáneamente:

* La cantidad de evaluaciones de la función.
* La precisión obtenida.
* El costo computacional requerido.

Mientras que Trapecio y Simpson necesitan una mayor cantidad de puntos para alcanzar una precisión aceptable, Gauss-Legendre consigue resultados más precisos utilizando una cantidad reducida de evaluaciones.

Esto demuestra que la selección óptima de nodos es más importante que simplemente aumentar el número de puntos de integración.

---

### **Conclusión**

Los resultados muestran que la **Cuadratura de Gauss-Legendre** es el método más eficiente entre los analizados. Su capacidad para seleccionar nodos óptimos permite alcanzar errores extremadamente pequeños con un costo computacional reducido, convirtiéndolo en una de las técnicas más potentes para la integración numérica de funciones suaves.



### **¿Por qué ocurrió esto?**

---

## <span style="color:#2E6599;">Preguntas de Análisis</span>

### **Pregunta 1: Eficiencia Computacional**

Explique la relación entre el número de evaluaciones de la función \(f(x)\) y el error absoluto obtenido por Gauss-Legendre frente a los métodos de Newton-Cotes.

¿Por qué se dice que Gauss optimiza el **costo computacional**?

### **Pregunta 2: El Efecto de la Oscilación**

Observe el comportamiento de la función en el intervalo dado.

¿Qué ocurre con la pendiente y la oscilación en las zonas de mayor curvatura?

Explique matemáticamente por qué los métodos compuestos tradicionales presentan mayores errores que la cuadratura de Gauss.

### **Pregunta 3: Límites del Método**

Si la función fuera:

**P(x) = 7x⁷ − 3x⁴ + 2x**

¿Cuántos nodos \(n\) requeriría Gauss-Legendre para obtener un error absoluto exactamente igual a cero?

**Sugerencia:** Justifique utilizando el teorema del grado de precisión máxima.

</div>

📘 ÍNDICE DE CAPÍTULOS


<span style="color:#D6004A">UNIDAD I</span>
<div style="background-color:#D6004A; color:white; padding:10px; font-size:20px; font-weight:bold;"> Integración Numérica: Métodos de Newton-Cotes y Cuadratura de Gauss </div>
<span style="color:#D6004A">CAPÍTULO 1</span>

Regla del Trapecio Compuesta

<span style="color:#D6004A">CAPÍTULO 2</span>

Regla de Simpson 1/3 Compuesta

<span style="color:#D6004A">CAPÍTULO 3</span>

Regla de Simpson 3/8 Compuesta

<span style="color:#D6004A">UNIDAD II</span>
<div style="background-color:#D6004A; color:white; padding:10px; font-size:20px; font-weight:bold;"> Cuadratura de Gauss-Legendre </div>
<span style="color:#D6004A">CAPÍTULO 4</span>

Obtención Algorítmica de Nodos y Pesos

<span style="color:#D6004A">CAPÍTULO 5</span>

Selección Automática del Número de Nodos

<span style="color:#D6004A">CAPÍTULO 6</span>

Análisis Comparativo de Precisión y Costo Computacional

# <span style="color:#D6004A">EJERCICIO 1 – CLASE DE LABORATORIO</span>

<div style="background-color:#D6004A;color:white;padding:10px;font-size:20px;font-weight:bold;">
Integración Numérica mediante Newton-Cotes y Cuadratura de Gauss-Legendre
</div>

---

## Objetivo

Desarrollar un script en Python capaz de calcular integrales definidas mediante:

- Regla del Trapecio Compuesta
- Regla de Simpson 1/3 Compuesta
- Regla de Simpson 3/8 Compuesta
- Cuadratura de Gauss-Legendre

El programa deberá permitir el ingreso de cualquier función desde teclado, calcular automáticamente la solución exacta y determinar el error absoluto de cada método.

---

## Integrales a evaluar

### Caso 1

\[
\int_0^3 \frac{1}{\sqrt{2\pi}}e^{-x^2/2}\,dx
\]

### Caso 2

\[
\int_0^2 \cos(x^2)\,dx
\]

### Caso 3

\[
\int_{-2}^{3}(x^5-2x^3+4)\,dx
\]

---

## Requisitos

### Trapecio Compuesto

Utilizar:

\[
N=12
\]

subintervalos.

### Simpson 1/3 Compuesto

Utilizar:

\[
N=12
\]

subintervalos.

### Simpson 3/8 Compuesto

Utilizar:

\[
N=12
\]

subintervalos.

### Gauss-Legendre

- Utilizar \(n=5\) nodos para los casos 1 y 2.
- Determinar automáticamente el número mínimo de nodos para el caso polinómico.
- Calcular algorítmicamente las raíces y pesos.

---

## Resultados esperados

Mostrar una tabla de la forma:

| Método | Puntos F(x) | Aproximación | Error Absoluto |
|----------|----------|----------|----------|
| Trapecio Compuesto | | | |
| Simpson 1/3 Compuesto | | | |
| Simpson 3/8 Compuesto | | | |
| Gauss-Legendre | | | |

---

## Preguntas de análisis

### 1. Eficiencia computacional

Explique la relación entre el número de evaluaciones de la función y el error obtenido.

### 2. Efecto de la oscilación

Analice el comportamiento de los métodos en regiones de alta curvatura.

### 3. Límite del método

Determine cuántos nodos requiere Gauss-Legendre para integrar exactamente:

\[
P(x)=7x^7-3x^4+2x
\]

utilizando el teorema del grado de precisión máxima.

# Desarrollo de la solución

[Markdown]
EJERCICIO 1 – CLASE DE LABORATORIO

[Código]
Importación de librerías

[Código]
Funciones Trapecio, Simpson 1/3, Simpson 3/8

[Código]
Gauss-Legendre algorítmico

[Código]
Detección automática de polinomio

[Código]
Programa principal

[Código]
Tabla final de resultados

[Markdown]
Análisis de resultados

[Markdown]
Respuestas de las preguntas